# 03 — Select, Filter & Column Transformations

**What you will learn:**
- `select()` and `selectExpr()` — choosing and computing columns
- `filter()` / `where()` — row-level filtering
- `withColumn()` — add or replace a column
- `withColumnRenamed()` / `toDF()` — renaming columns
- `drop()` — removing columns
- `distinct()` / `dropDuplicates()` — deduplication
- `limit()` and `sample()` — controlling row count
- `cast()` — changing column data types
- `orderBy()` / `sort()` — sorting rows

## Setup — Reusable Sample DataFrame

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, lit, expr, when, upper, lower, trim

schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("hire_date",  StringType(),  True),
    StructField("manager_id", IntegerType(), True),
])

data = [
    (1,  "Alice",   "Engineering", 95000.0,  "2022-01-15", 10),
    (2,  "Bob",     "Marketing",   72000.0,  "2021-06-01", 11),
    (3,  "Charlie", "Engineering", 105000.0, "2020-03-20", 10),
    (4,  "Diana",   "HR",          68000.0,  "2023-09-10", 12),
    (5,  "Eve",     "Marketing",   78000.0,  "2022-11-05", 11),
    (6,  "Frank",   "Engineering", 88000.0,  "2021-04-12", 10),
    (7,  "Grace",   "HR",          71000.0,  "2022-07-30", 12),
    (8,  "Hank",   "Engineering",  None,     "2023-01-01", 10),
    (9,  "Ivy",    "Marketing",    72000.0,  "2021-06-01", 11),  # duplicate salary+dept as Bob
]

df = spark.createDataFrame(data, schema=schema)
df.show()

## 1. select() — Choose Columns

In [ ]:
# Select by name (string)
df.select("emp_id", "name", "department").show()

In [ ]:
# Select using col() object — preferred for expressions
df.select(col("emp_id"), col("name"), col("salary")).show()

In [ ]:
# Select with alias — rename a column in the output
df.select(
    col("emp_id").alias("id"),
    col("name").alias("employee_name"),
    col("salary")
).show()

In [ ]:
# Compute a new column in select
df.select(
    col("name"),
    col("salary"),
    (col("salary") * 1.10).alias("salary_after_10pct_raise")
).show()

## 2. selectExpr() — SQL Expressions as Strings

In [ ]:
# selectExpr accepts SQL string expressions — handy for quick transforms
df.selectExpr(
    "emp_id",
    "upper(name) as name_upper",
    "salary",
    "salary * 1.10 as salary_raised",
    "case when salary > 90000 then 'Senior' else 'Junior' end as level"
).show()

## 3. filter() and where()

`filter()` and `where()` are identical — both filter rows.
You can pass a Column expression or a SQL string.

In [ ]:
# Filter using column expression
df.filter(col("salary") > 80000).show()

In [ ]:
# Filter using SQL string (where is an alias for filter)
df.where("department = 'Engineering'").show()

In [ ]:
# Compound conditions — use & (AND), | (OR), ~ (NOT)
# Always wrap each condition in parentheses with bitwise operators
df.filter(
    (col("department") == "Engineering") & (col("salary") > 90000)
).show()

In [ ]:
df.filter(
    (col("department") == "Marketing") | (col("department") == "HR")
).show()

In [ ]:
# Filter using isin()
df.filter(col("department").isin("Engineering", "HR")).show()

In [ ]:
# Filter using between()
df.filter(col("salary").between(70000, 90000)).show()

In [ ]:
# Filter nulls
df.filter(col("salary").isNull()).show()
df.filter(col("salary").isNotNull()).show()

## 4. withColumn() — Add or Replace a Column

In [ ]:
# Add a new column (constant value)
df.withColumn("currency", lit("USD")).show()

In [ ]:
# Add a column computed from existing columns
df2 = df.withColumn("annual_bonus", col("salary") * 0.15)
df2.show()

In [ ]:
# Replace an existing column (same name)
df3 = df.withColumn("salary", (col("salary") / 1000).alias("salary_k"))
df3.show()  # salary is now in thousands

In [ ]:
# withColumn with when/otherwise (if-else logic)
df4 = df.withColumn(
    "salary_band",
    when(col("salary") >= 100000, "Band A")
    .when(col("salary") >= 80000,  "Band B")
    .when(col("salary") >= 70000,  "Band C")
    .otherwise("Band D")
)
df4.select("name", "salary", "salary_band").show()

## 5. withColumnRenamed() and toDF()

In [ ]:
# Rename a single column
df.withColumnRenamed("hire_date", "joining_date").show(3)

In [ ]:
# Rename all columns at once using toDF()
new_names = ["id", "employee", "dept", "pay", "joined", "mgr_id"]
df.toDF(*new_names).show(3)

## 6. drop() — Remove Columns

In [ ]:
# Drop a single column
df.drop("manager_id").show(3)

In [ ]:
# Drop multiple columns
df.drop("manager_id", "hire_date").show(3)

## 7. distinct() and dropDuplicates()

In [ ]:
# Add some duplicate rows
dup_data = data + [
    (2, "Bob", "Marketing", 72000.0, "2021-06-01", 11),   # exact duplicate
    (9, "Ivy", "Marketing", 72000.0, "2021-06-01", 11),   # different emp_id, same rest
]
df_dup = spark.createDataFrame(dup_data, schema=schema)
print("With duplicates:", df_dup.count())

# distinct() — remove fully identical rows
df_distinct = df_dup.distinct()
print("After distinct() :", df_distinct.count())

In [ ]:
# dropDuplicates() — remove rows with duplicate values in specific columns
df_dedup = df_dup.dropDuplicates(["name", "department", "salary"])
print("After dropDuplicates(name,dept,salary):", df_dedup.count())
df_dedup.show()

## 8. limit() and sample()

In [ ]:
# limit — first N rows (deterministic)
df.limit(3).show()

In [ ]:
# sample — random fraction of rows (non-deterministic unless seed is fixed)
df_sample = df.sample(fraction=0.5, seed=42)
df_sample.show()

## 9. cast() — Change Data Types

In [ ]:
from pyspark.sql.types import DateType, LongType

# Cast hire_date string to DateType
df_cast = df.withColumn("hire_date", col("hire_date").cast(DateType()))
df_cast.printSchema()
df_cast.show()

In [ ]:
# Cast salary from Double to Long (integer)
df.withColumn("salary_int", col("salary").cast(LongType())).show(3)

## 10. orderBy() / sort()

In [ ]:
# Sort ascending (default)
df.orderBy("salary").show()

In [ ]:
# Sort descending
df.orderBy(col("salary").desc()).show()

In [ ]:
# Sort by multiple columns
df.orderBy(col("department").asc(), col("salary").desc()).show()

## 11. Chaining Transformations

In [ ]:
# Transformations are lazy — chaining returns a new DataFrame plan
result = (
    df
    .filter(col("department") == "Engineering")
    .withColumn("salary_k", (col("salary") / 1000).cast("double"))
    .withColumn("level", when(col("salary") >= 100000, "Senior").otherwise("Mid"))
    .select("emp_id", "name", "salary_k", "level")
    .orderBy(col("salary_k").desc())
)
result.show()

## Key Takeaways

| Operation | Method | Notes |
|---|---|---|
| Select columns | `select("a","b")` or `select(col("a"))` | Use `col()` for expressions |
| SQL expressions | `selectExpr("upper(name) as n")` | String SQL inside Python |
| Filter rows | `filter()` / `where()` | Both identical |
| Compound filter | `(cond1) & (cond2)` | Always parenthesize each condition |
| Add/replace column | `withColumn(name, expr)` | Returns a new DataFrame |
| Rename column | `withColumnRenamed(old, new)` | |
| Rename all | `toDF(*new_names)` | |
| Remove column | `drop(col)` | |
| Dedup all cols | `distinct()` | |
| Dedup subset | `dropDuplicates([cols])` | |
| Change type | `col.cast(Type())` | |
| Sort | `orderBy(col.desc())` | |